In [1]:
import numpy as np
from aeon.datasets import load_classification

In [2]:
def generate_folds_dynamic(
    X,
    y,
    max_splits=None,
    min_splits=2,
    random_state=12,
    require_all_classes=True,
    n_splits=None,
):
    from sklearn.model_selection import StratifiedKFold

    y = np.asarray(y)
    classes, counts = np.unique(y, return_counts=True)
    all_classes = set(classes)

    if n_splits is not None:
        if max_splits is not None:
            raise ValueError("Pass either n_splits or max_splits, not both.")
        max_splits = n_splits

    if min_splits < 2:
        raise ValueError(f"min_splits must be >= 2, got {min_splits}")

    n_samples = len(y)
    n_classes = len(classes)

    if require_all_classes:
        max_by_class = int(counts.min())
        max_by_val_size = n_samples // n_classes
        max_feasible = min(max_by_class, max_by_val_size)
        if max_feasible < min_splits:
            raise ValueError(
                f"No valid split in [{min_splits}, ...] with all classes in every validation fold. "
                f"n_samples={n_samples}, n_classes={n_classes}, min_class_count={counts.min()}"
            )
    else:
        max_feasible = n_samples

    if max_splits is None:
        start_splits = max_feasible
    else:
        start_splits = min(max_splits, n_samples)

    if start_splits < min_splits:
        raise ValueError(
            f"Invalid range: start_splits={start_splits} < min_splits={min_splits}."
        )

    for current_splits in range(start_splits, min_splits - 1, -1):
        try:
            skf = StratifiedKFold(n_splits=current_splits, shuffle=True, random_state=random_state)
            folds = [(train.tolist(), val.tolist()) for train, val in skf.split(X, y)]
        except ValueError:
            if current_splits > min_splits:
                print(
                    f"StratifiedKFold failed with n_splits={current_splits}, "
                    f"retrying with n_splits={current_splits - 1}"
                )
            continue

        if require_all_classes:
            if not all(set(y[val_idx]) == all_classes for _, val_idx in folds):
                if current_splits > min_splits:
                    print(
                        f"Validation folds missing classes with n_splits={current_splits}, "
                        f"retrying with n_splits={current_splits - 1}"
                    )
                continue

        return folds

    raise ValueError(
        f"Could not create folds for any n_splits in range [{min_splits}, {start_splits}]."
    )


In [3]:
X_train, y_train = load_classification("FaceFour", split="train")
np.unique(y_train)

array(['1', '2', '3', '4'], dtype='<U1')

In [4]:
folds = generate_folds_dynamic(X_train, y_train, max_splits=30, min_splits=2, random_state=12)

all_classes = set(np.unique(y_train))
print(f"Got {len(folds)} folds")
for i, (train_idx, val_idx) in enumerate(folds):
    val_classes = set(np.array(y_train)[val_idx])
    ok = val_classes == all_classes
    print(f"  Fold {i}: val_size={len(val_idx)}, classes={val_classes}, all_classes_present={ok}")

StratifiedKFold failed with n_splits=24, retrying with n_splits=23
StratifiedKFold failed with n_splits=23, retrying with n_splits=22
StratifiedKFold failed with n_splits=22, retrying with n_splits=21
StratifiedKFold failed with n_splits=21, retrying with n_splits=20
StratifiedKFold failed with n_splits=20, retrying with n_splits=19
StratifiedKFold failed with n_splits=19, retrying with n_splits=18
StratifiedKFold failed with n_splits=18, retrying with n_splits=17
StratifiedKFold failed with n_splits=17, retrying with n_splits=16
StratifiedKFold failed with n_splits=16, retrying with n_splits=15
StratifiedKFold failed with n_splits=15, retrying with n_splits=14
StratifiedKFold failed with n_splits=14, retrying with n_splits=13
StratifiedKFold failed with n_splits=13, retrying with n_splits=12
StratifiedKFold failed with n_splits=12, retrying with n_splits=11
StratifiedKFold failed with n_splits=11, retrying with n_splits=10
StratifiedKFold failed with n_splits=10, retrying with n_split

/home/petelin/TSCGlue/.venv/lib/python3.13/site-packages/sklearn/model_selection/_split.py:811: UserWarning: The least populated class in y has only 3 members, which is less than n_splits=8.
  warnings.warn(
/home/petelin/TSCGlue/.venv/lib/python3.13/site-packages/sklearn/model_selection/_split.py:811: UserWarning: The least populated class in y has only 3 members, which is less than n_splits=7.
  warnings.warn(
/home/petelin/TSCGlue/.venv/lib/python3.13/site-packages/sklearn/model_selection/_split.py:811: UserWarning: The least populated class in y has only 3 members, which is less than n_splits=6.
  warnings.warn(
/home/petelin/TSCGlue/.venv/lib/python3.13/site-packages/sklearn/model_selection/_split.py:811: UserWarning: The least populated class in y has only 3 members, which is less than n_splits=5.
  warnings.warn(
/home/petelin/TSCGlue/.venv/lib/python3.13/site-packages/sklearn/model_selection/_split.py:811: UserWarning: The least populated class in y has only 3 members, which is

In [ ]:
from pathlib import Path

import pandas as pd

data_dir = Path("data") if Path("data").exists() else Path("..") / "data"
dataset_names = sorted([p.name for p in data_dir.iterdir() if p.is_dir()])
rows = []

for dataset_name in dataset_names:
    try:
        _, y_train = load_classification(dataset_name, split="train")
        y_train = np.asarray(y_train)
        classes, counts = np.unique(y_train, return_counts=True)

        min_class_count = int(counts.min())
        leave_instance_out_feasible = min_class_count >= 2

        rows.append({
            "dataset": dataset_name,
            "n_train": int(len(y_train)),
            "n_classes": int(len(classes)),
            "min_class_count": min_class_count,
            "leave_instance_out_feasible": leave_instance_out_feasible,
            "reason": "ok" if leave_instance_out_feasible else "has class with <2 train samples",
        })
    except Exception as e:
        rows.append({
            "dataset": dataset_name,
            "n_train": np.nan,
            "n_classes": np.nan,
            "min_class_count": np.nan,
            "leave_instance_out_feasible": False,
            "reason": f"load_failed: {e}",
        })

loo_feasibility = pd.DataFrame(rows).sort_values(["leave_instance_out_feasible", "min_class_count", "dataset"]).reset_index(drop=True)

print(f"Datasets checked: {len(loo_feasibility)}")
print(f"Feasible for leave-instance-out: {int(loo_feasibility['leave_instance_out_feasible'].sum())}")
print(f"Not feasible: {int((~loo_feasibility['leave_instance_out_feasible']).sum())}")

loo_feasibility
